In [2]:
library(dplyr)  
library(tidyr)

df <- read.delim("~/code_athon/results/all_finemapped_variant_gene_pairs.tsv")   # or read.csv(...) / readr::read_tsv(...)

out <- df %>%  
  distinct(variant_key) %>%          # same variant can appear for several genes  
  separate(variant_key,  
           into = c("chr_name", "chrom_start", "ref", "alt"),  
           sep = ":", remove = TRUE, extra = "drop") %>%  
  mutate(locus = paste0("chr", chr_name, ":", chrom_start))

#write.table(out,  
 #           "~/code_athon/results/variant_key_coordinates.csv",  
  #          sep = "\t", quote = FALSE, row.names = FALSE)  


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
dim(unique(out))
dim(out)

[1] 336351      5

[1] 336351      5

In [4]:
head(df)
dim(df)

,gene_id,variant_key,MAGE_PIP,MAGE_symbol,GTEx_PIP,GTEx_symbol,status,MAGE_highPIP,GTEx_highPIP,Both_highPIP
,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ENSG00000000457,1:169833619:G:T,0.961535914,SCYL3,NA,,MAGE_finemapped_only,True,False,False
2,ENSG00000001036,6:143416357:T:TA,0.013926998,FUCA2,NA,,MAGE_finemapped_only,False,False,False
3,ENSG00000001036,6:143418884:T:A,0.008913702,FUCA2,NA,,MAGE_finemapped_only,False,False,False
4,ENSG00000001036,6:143422225:A:G,0.008913702,FUCA2,NA,,MAGE_finemapped_only,False,False,False
5,ENSG00000001036,6:143425557:A:G,0.008913702,FUCA2,NA,,MAGE_finemapped_only,False,False,False
6,ENSG00000001036,6:143431724:T:G,0.007978768,FUCA2,NA,,MAGE_finemapped_only,False,False,False


[1] 452748     10

In [5]:
head(out)
dim(out)
length(unique(df$variant_key))

,chr_name,chrom_start,ref,alt,locus
,<chr>,<chr>,<chr>,<chr>,<chr>
1,1,169833619,G,T,chr1:169833619
2,6,143416357,T,TA,chr6:143416357
3,6,143418884,T,A,chr6:143418884
4,6,143422225,A,G,chr6:143422225
5,6,143425557,A,G,chr6:143425557
6,6,143431724,T,G,chr6:143431724


[1] 336351      5

[1] 336351

In [8]:
library(dplyr)

variants_all <- out %>%  
  mutate(variant_key = paste(chr_name, chrom_start, ref, alt, sep = ":")) %>%  
  left_join(df %>% select(variant_key, MAGE_PIP, GTEx_PIP),  
            by = "variant_key") %>%  
  mutate(  
    present_in = case_when(  
      !is.na(MAGE_PIP) & !is.na(GTEx_PIP) ~ "both",  
      !is.na(MAGE_PIP)                    ~ "mage",  
      !is.na(GTEx_PIP)                    ~ "gtex",  
      TRUE                                ~ NA_character_  
    ),  
    alleles = paste0('["', ref, '","', alt, '"]')  
  ) %>%  
  select(locus, alleles, present_in)

head(variants_all)  
dim(variants_all)                       # should be 336351 x 3  
table(variants_all$present_in, useNA = "ifany")  

,locus,alleles,present_in
,<chr>,<chr>,<chr>
1,chr1:169833619,"[""G"",""T""]",mage
2,chr6:143416357,"[""T"",""TA""]",mage
3,chr6:143418884,"[""T"",""A""]",mage
4,chr6:143422225,"[""A"",""G""]",mage
5,chr6:143425557,"[""A"",""G""]",mage
6,chr6:143431724,"[""T"",""G""]",mage


[1] 452748      3


  both   gtex   mage 
 48701 199252 204795 

In [12]:
dim(unique(variants_all[,c(1,2,3)]))

[1] 374800      3

Save

In [13]:
write.csv(variants_all, "~/code_athon/results/variant_annotations.csv",
         row.names=FALSE)